# HOROv1 — Pipeline ETL

**Análise de Vento para Aeródromos (RBAC154)**

Desenvolvido por John Heberty de Freitas — john.7heberty@gmail.com

---

## Instruções

1. Coloque os arquivos CSV em `data/raw/`
2. Execute as células na ordem
3. Os resultados estarão em `data/gold/exports/`

Para execução via terminal:
```bash
python orchestrator.py --all
```

In [ ]:
# Célula 1 — Instalar / verificar dependências
%pip install -q pandas numpy windrose matplotlib scikit-learn opencv-python moviepy selenium webdriver-manager Unidecode chardet python-dateutil fastparquet pyarrow

In [ ]:
# Célula 2 — Importações
import os
import sys

# Garante que o diretório raiz do projeto está no path
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.core.config import cfg
from pipeline.core.logger import configure_logging
from pipeline.core.models import PipelineContext

configure_logging(level='INFO')
cfg.ensure_dirs()
print('Ambiente configurado.')
print(f'  data/raw:    {cfg.output.data_raw}')
print(f'  data/bronze: {cfg.output.data_bronze}')
print(f'  data/silver: {cfg.output.data_silver}')
print(f'  data/gold:   {cfg.output.data_gold}')

In [ ]:
# Célula 3 — Verificar arquivos de entrada
input_files = cfg.output.input_csvs()
print(f'Arquivos encontrados em data/raw/: {len(input_files)}')
for f in input_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
# Célula 4 — Inicializar contexto do pipeline
context = PipelineContext(input_files=input_files)
print(f'Pipeline run_id: {context.run_id}')
print(f'Iniciado em:     {context.started_at}')

In [ ]:
# Célula 5 — Stage 1: Ingest (Raw → Bronze)
from pipeline.stages import s01_ingest
context = s01_ingest.run(context, cfg)

print('\n--- Resumo Bronze ---')
for station, rec in context.bronze.items():
    status = 'REJEITADO' if rec.rejected else 'OK'
    print(f'  {station}: {status}  ({rec.row_count} linhas)')
    if rec.rejected:
        print(f'    Motivo: {rec.rejection_reason}')

In [ ]:
# Célula 6 — Stage 2: Validate
from pipeline.stages import s02_validate
context = s02_validate.run(context, cfg)

In [ ]:
# Célula 7 — Stage 3: Transform (Bronze → Silver)
from pipeline.stages import s03_transform
context = s03_transform.run(context, cfg)

print('\n--- Resumo Silver ---')
for station, rec in context.silver.items():
    years = rec.years_available
    print(f'  {station}: {len(rec.df)} registros | anos {min(years)}–{max(years)}')

In [ ]:
# Célula 8 — Stage 4: Analyze (Wind tables)
from pipeline.stages import s04_analyze
context = s04_analyze.run(context, cfg)

print('\n--- Tabelas de vento calculadas ---')
for station, by_years in context.wind_tables.items():
    for years, wt in by_years.items():
        print(f'  {station} / {years}y: {wt.total_records} obs | calmas {wt.calm_pct:.1f}%')

In [ ]:
# Célula 9 — Stage 5: Enrich (Declinação magnética)
# REQUER BROWSER CHROME CONFIGURADO
from pipeline.stages import s05_enrich
context = s05_enrich.run(context, cfg)

print('\n--- Declinações ---')
for station, rec in context.silver.items():
    print(f'  {station}: {rec.magnetic_declination:.4f}°')

In [ ]:
# Célula 10 — Stage 6: Optimize (Melhor orientação de pista)
from pipeline.stages import s06_optimize
context = s06_optimize.run(context, cfg)

print('\n--- Resultados de Otimização ---')
for station, by_years in context.results.items():
    for years, res in by_years.items():
        print(f'  {station} / {years} anos:')
        print(f'    Melhor pista: {res.runway_designation}  |  Rumo: {res.best_heading_deg:.0f}°')
        print(f'    FO: {res.fo_pct:.1f}%  |  Vento través: {res.crosswind_pct:.1f}%')

In [ ]:
# Célula 11 — Stage 7: Export (Vídeo, GIF, JSON final)
from pipeline.stages import s07_export
context = s07_export.run(context, cfg)

print(f'\nPipeline finalizado! Run ID: {context.run_id}')
print(f'Estágios executados: {context.stages_executed}')
print(f'Saídas em: {cfg.output.data_gold}')